<div style="padding:48px 42px;border:1px solid #d7e2ee;border-radius:10px;background:#f8fbff;">
<h1 style="font-size:2.3em;font-weight:800;margin:0 0 12px;">Hyperloom Workshop</h1>
<h2 style="font-size:1.24em;font-weight:500;color:#1b5e8f;margin:0 0 22px;">Agentic inference optimization for Qwen3-30B-A3B on AMD MI300X</h2>
<hr style="border:0;border-top:1px solid #d7e2ee;margin:18px 0;">
<p style="margin:0;color:#384454;font-size:1.02em;">Runtime: vLLM | Kernel backend: Forge | Workload: CONC=16, ISL=1024, OSL=1024 | Target gain: 10%</p>
</div>

## Workshop Roadmap

This notebook demonstrates how Hyperloom runs an evidence-driven optimization loop for an LLM serving workload. It uses one example run as the data source, then finishes with the exact notebook cells needed to launch and monitor a fresh run.

| # | Topic | What to look for |
|---|-------|------------------|
| 1 | Hyperloom architecture | How the coordinator, agents, profilers, and kernel backends fit together |
| 2 | Optimization loop | Baseline -> profile -> tune -> validate -> report |
| 3 | Example run | Qwen3-30B-A3B on vLLM, MI300X, CONC=16 |
| 4 | Performance gain | Baseline, best variant, validated gain, and target status |
| 5 | Why the gain happens | Memory-bound MoE/GEMM bottleneck and Forge-selected vLLM tuned configs |
| 6 | Live run cells | Startup script cell and monitor cell |

The notebook intentionally treats terminal logs as secondary. The run state, benchmark artifacts, and final report are the source of truth.

---
## 1. Hyperloom Architecture

Hyperloom treats inference optimization as a measured search problem. The user provides a model, runtime, hardware target, and workload. Hyperloom then coordinates agents and tools that repeatedly propose one change, benchmark it, and keep only validated wins.

![Hyperloom loop](slides/hyperloom_loop.png)

![Simplified Hyperloom architecture](slides/simplified_fig1.png)

| Layer | Role in the workflow |
|-------|----------------------|
| `inference_optimizer` / Coordinator | Starts the run, creates the session, owns phase sequencing, action execution, policy gates, and final reporting. |
| Orchestration agent | Plans the next allowed action and adapts from benchmark results. |
| Kernel agent | Routes hot kernels into TraceLens analysis and kernel optimization backends. |
| Critic agent | Reviews risky source or kernel changes before they are promoted. |
| Robustness agent | Watches stalls, crashes, process health, and recovery signals. |
| Framework agent | Discovers relevant vLLM/SGLang framework-source candidates when that phase is enabled. |
| Magpie / InferenceX | Launches serving benchmarks and records throughput evidence for vLLM/SGLang/Atom workloads. |
| TraceLens / roofline model | Converts traces and model metadata into bottleneck direction, kernel candidates, and performance ceilings. |
| Forge / GEAK / OOB backends | Tune or rewrite kernels, then return a candidate that still must pass end-to-end validation. |

The important design point is that Hyperloom does not accept a microbenchmark result by itself. A candidate becomes the current best only after it improves the actual serving workload under the same measurement path.

---
## 2. Optimization Loop

The live runtime phase chain is:

```text
PRELUDE -> FRAMEWORK_PR -> EXPLORE -> KERNEL -> SWEEP -> CLOSE
```

| Phase | Purpose |
|-------|---------|
| PRELUDE | Establish target comparison, measure the baseline, and collect the first profile or roofline signal. |
| FRAMEWORK_PR | Optionally test framework-source candidates discovered by the framework agent. |
| EXPLORE | Search serving parameters, environment knobs, and source-patch ideas. |
| KERNEL | Route hot kernels from TraceLens into Forge, GEAK, or other kernel backends. |
| SWEEP | Check that the optimized stack still wins across workload frontiers. |
| CLOSE | Write the final report and machine-readable run summary. |

A full session records `manifest.json`, `state.json`, per-action `runs/`, agent workspaces, reports, and, when closeout completes normally, `session_breakdown.json`. For dashboards, `session_breakdown.json` is the stable external contract. This example uses `state.json` and raw benchmark artifacts because the run ended at the wall-time limit and the safety-net report path was used.

---
## 3. Example Run Configuration

The example run uses a vLLM serving workload for Qwen3-30B-A3B:

| Item | Value |
|------|-------|
| Model | `Qwen-Qwen3-30B-A3B` |
| Framework | `vllm` |
| GPU type | `mi300x` |
| Tensor parallelism | `TP=1` |
| Concurrency | `CONC=16` |
| Prompt / output length | `ISL=1024`, `OSL=1024` |
| Precision | `bf16` |
| Target | `10%` throughput gain |
| Time budget | `1 hour` |
| Winning backend | `Forge` |

Set `HYPERLOOM_EXAMPLE_SESSION_DIR` if the example artifacts are stored outside the standard Hyperloom workspace roots.

In [ ]:
from pathlib import Path
import json
import os
from pprint import pprint


EMBEDDED_EXAMPLE_SUMMARY = {
    "framework": "vllm",
    "concurrency": 16,
    "sequence_lengths": {"isl": 1024, "osl": 1024},
    "baseline_tok_s_per_gpu": 1375.208,
    "hot_baseline_tok_s_per_gpu": 1441.083,
    "best_tok_s_per_gpu": 1454.510,
    "validated_gain_pct": 5.7665,
    "gain_vs_hot_baseline_pct": 0.9317,
    "best_action": "gemm_tuning",
    "best_backend": "forge",
    "best_variant": "vllm_moe_triton",
    "stop_reason": "example_run_complete",
    "roofline_bound": "memory",
    "within_roofline_pct": 68.5,
    "gap_to_roofline_pct": 31.5,
    "top_ops": [
        {
            "name": "moe_fused",
            "pct_time": 88.41,
            "bound": "memory",
            "arithmetic_intensity": 1.55,
        },
        {
            "name": "attention_decode",
            "pct_time": 5.62,
            "bound": "memory",
            "arithmetic_intensity": 0.92,
        },
        {
            "name": "sampling_and_logits",
            "pct_time": 2.48,
            "bound": "memory",
            "arithmetic_intensity": 0.76,
        },
    ],
    "report_exists": False,
    "current_setting_exists": False,
    "session_breakdown_exists": False,
    "data_source": "embedded_example",
}


def find_example_session():
    override = os.environ.get("HYPERLOOM_EXAMPLE_SESSION_DIR")
    if override:
        path = Path(override)
        if not (path / "state.json").exists():
            raise FileNotFoundError(f"HYPERLOOM_EXAMPLE_SESSION_DIR has no state.json: {path}")
        return path

    patterns = [
        "/workspace/hyperloom*/Qwen-Qwen3-30B-A3B/*/state.json",
        "/workspace/hyperloom*/Qwen3-30B-A3B/*/state.json",
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(Path("/").glob(pattern.lstrip("/")))
    candidates = sorted(candidates, key=lambda path: path.stat().st_mtime, reverse=True)
    if not candidates:
        return None
    return candidates[0].parent


EXAMPLE_SESSION_DIR = find_example_session()

if EXAMPLE_SESSION_DIR is None:
    summary = dict(EMBEDDED_EXAMPLE_SUMMARY)
else:
    STATE_PATH = EXAMPLE_SESSION_DIR / "state.json"
    REPORT_PATH = EXAMPLE_SESSION_DIR / "reports" / "final.md"
    CURRENT_SETTING_PATH = EXAMPLE_SESSION_DIR / "current_setting.sh"
    SESSION_BREAKDOWN_PATH = EXAMPLE_SESSION_DIR / "session_breakdown.json"

    state = json.loads(STATE_PATH.read_text())
    current_best = state.get("current_best") or {}
    roofline = state.get("baseline_roofline_ceiling") or {}
    perfmodel = roofline.get("perfmodel_breakdown") or {}
    ops = sorted(perfmodel.get("ops") or [], key=lambda op: op.get("pct_time", 0), reverse=True)

    baseline = float(state.get("baseline_tput") or 0.0)
    hot_baseline = float(state.get("baseline_hot_tput") or 0.0)
    best = float(current_best.get("tput") or 0.0)
    gain = ((best / baseline) - 1.0) * 100.0 if baseline else None
    hot_gain = ((best / hot_baseline) - 1.0) * 100.0 if hot_baseline else None

    summary = {
        "framework": state.get("framework"),
        "concurrency": state.get("conc"),
        "sequence_lengths": {"isl": state.get("isl"), "osl": state.get("osl")},
        "baseline_tok_s_per_gpu": baseline,
        "hot_baseline_tok_s_per_gpu": hot_baseline,
        "best_tok_s_per_gpu": best,
        "validated_gain_pct": state.get("cumulative_gain_validated", gain),
        "gain_vs_hot_baseline_pct": hot_gain,
        "best_action": current_best.get("action"),
        "best_backend": current_best.get("engine"),
        "best_variant": current_best.get("variant_name"),
        "stop_reason": state.get("stop_reason"),
        "roofline_bound": roofline.get("roofline_bound_kind"),
        "within_roofline_pct": roofline.get("within_roofline_pct"),
        "gap_to_roofline_pct": roofline.get("gap_to_roofline_pct"),
        "top_ops": [
            {
                "name": op.get("name"),
                "pct_time": round(float(op.get("pct_time", 0)) * 100, 2),
                "bound": op.get("bound"),
                "arithmetic_intensity": round(float(op.get("ai", 0)), 2),
            }
            for op in ops[:5]
        ],
        "report_exists": REPORT_PATH.exists(),
        "current_setting_exists": CURRENT_SETTING_PATH.exists(),
        "session_breakdown_exists": SESSION_BREAKDOWN_PATH.exists(),
        "data_source": str(EXAMPLE_SESSION_DIR),
    }

pprint(summary)

---
## 4. Performance Gain

The example run reached a validated 5%+ throughput gain.

<div style="display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:16px;margin:14px 0 18px;">
<div style="background:#f6f8ff;border:1px solid #dce5ff;border-radius:8px;padding:18px;"><div style="font-size:1.45em;font-weight:800;color:#2b6cb0;">1375.2</div><div style="font-weight:700;color:#202938;margin-top:4px;">baseline tok/s/GPU</div><div style="color:#5b6678;margin-top:6px;">gain accounting anchor</div></div>
<div style="background:#f4fff7;border:1px solid #ccebd6;border-radius:8px;padding:18px;"><div style="font-size:1.45em;font-weight:800;color:#2f855a;">1454.5</div><div style="font-weight:700;color:#202938;margin-top:4px;">best tok/s/GPU</div><div style="color:#5b6678;margin-top:6px;">Forge GEMM tuned</div></div>
<div style="background:#fff8ef;border:1px solid #f7d7ad;border-radius:8px;padding:18px;"><div style="font-size:1.45em;font-weight:800;color:#c05621;">+5.77%</div><div style="font-weight:700;color:#202938;margin-top:4px;">validated gain</div><div style="color:#5b6678;margin-top:6px;">5%+ gain reached</div></div>
</div>

The winning action was `gemm_tuning` with the `forge` backend and `vllm_moe_triton` tuner. Hyperloom promoted it because the tuned vLLM configuration improved the same end-to-end serving benchmark used for gain accounting.

A stricter comparison against the later hot baseline measurement, `1441.1 tok/s/GPU`, gives a smaller `+0.93%` delta. The notebook reports Hyperloom's validated run-state gain because that is the value used by the optimizer's promotion and target accounting.

In [ ]:
baseline = float(summary["baseline_tok_s_per_gpu"])
hot_baseline = float(summary["hot_baseline_tok_s_per_gpu"] or 0)
best = float(summary["best_tok_s_per_gpu"])
gain_pct = float(summary["validated_gain_pct"])
top_ops = summary["top_ops"]

try:
    import matplotlib.pyplot as plt

    labels = ["Baseline", "Hot baseline", "Best"] if hot_baseline else ["Baseline", "Best"]
    values = [baseline, hot_baseline, best] if hot_baseline else [baseline, best]
    colors = ["#2b6cb0", "#718096", "#2f855a"] if hot_baseline else ["#2b6cb0", "#2f855a"]

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].bar(labels, values, color=colors)
    axes[0].set_ylabel("tok/s/GPU")
    axes[0].set_title(f"End-to-end throughput (+{gain_pct:.2f}% run-state gain)")
    for i, value in enumerate(values):
        axes[0].text(i, value, f"{value:.1f}", ha="center", va="bottom")

    names = [op["name"] for op in top_ops]
    times = [op["pct_time"] for op in top_ops]
    axes[1].barh(names[::-1], times[::-1], color="#805ad5")
    axes[1].set_xlabel("Estimated decode time share (%)")
    axes[1].set_title("Top modeled bottlenecks")

    plt.tight_layout()
    plt.show()
except Exception as exc:
    print(f"matplotlib chart skipped: {exc}")
    print(f"Baseline: {baseline:.2f} tok/s/GPU")
    if hot_baseline:
        print(f"Hot baseline: {hot_baseline:.2f} tok/s/GPU")
    print(f"Best: {best:.2f} tok/s/GPU")
    print(f"Gain: {gain_pct:.2f}%")
    print("Top ops:")
    for op in top_ops:
        print(f"  {op['name']}: {op['pct_time']:.2f}% time, bound={op['bound']}, AI={op['arithmetic_intensity']}")

---
## 5. Why This Run Gets Faster

The roofline estimate classifies the baseline as **memory-bound**. At the baseline anchor, the workload reaches about **68.5%** of the modeled memory roofline, leaving a **31.5%** gap to the estimated ceiling. The compute ceiling is far above the memory ceiling, so better data movement and kernel selection matter more than simply chasing more FLOPs.

The modeled decode breakdown points to one dominant target:

| Operation | Time share | Bound | Why it matters |
|-----------|------------|-------|----------------|
| `moe_fused` | `88.41%` | memory | Dominates decode time and moves about `37.34 GB` in the model estimate. |
| `sdpa` | `5.75%` | memory | Secondary attention cost. |
| `q_proj` / `o_proj` | `1.93%` each | memory | Projection work, much smaller than MoE. |
| `lm_head` | `1.49%` | memory | Output projection cost. |

Trace evidence points in the same direction: MoE GEMM kernels are among the top GPU consumers, including `ck::kernel_moe_gemm` entries at roughly `22.19%` and `12.23%` GPU time. That makes `vllm_moe_triton` GEMM tuning the natural first high-leverage action.

Forge tested `10` M-buckets for the vLLM MoE Triton path. The tuning artifact reports `best_micro_speedup=1.2976` and `avg_micro_speedup=1.0796`; the largest bucket-level win was at `M=48`, improving from `475.50 us` to `366.43 us`. The retained tuned config targets the measured hardware/model shape:

```text
E=128, N=768, device=AMD Instinct MI300X, dtype=bfloat16
```

The end-to-end gain is smaller than the best microbenchmark speedup because only part of serving time is affected, the workload remains memory-bound, and other runtime overheads stay unchanged. The result is still useful: Hyperloom identified the dominant MoE/GEMM path, generated a tuned vLLM config for that path, and validated a serving-throughput gain under the same workload.

---
## 6. Reproducibility Artifacts

A Hyperloom session leaves behind artifacts that make the result auditable:

| Artifact | Purpose |
|----------|---------|
| `state.json` | Current phase, baseline, current best, gain, stop reason, and roofline summary. |
| `manifest.json` | Run identity, workload, framework, and session metadata. |
| `session_breakdown.json` | Stable dashboard/reporting contract when CLOSE completes normally. |
| `runs/` | Per-action workspaces for baseline, profiling, tuning, sweep, and validation. |
| `reports/final.md` | Human-readable closeout report. |
| `current_setting.sh` | Generated launch recipe for the best known setting. |

For this example, use `state.json` plus the raw throughput JSONs as the metric source of truth. The final report was produced by the safety-net closeout path because the run hit the time budget. The throughput benchmark files were written and parsed, while the downstream accuracy eval path exited nonzero after throughput collection, so present this example as **throughput evidence**, not as a complete accuracy-validation report.

The key integration artifact is the `VLLM_TUNED_CONFIG_FOLDER` recorded in `current_best.extra_envs`. It points at the tuned vLLM MoE Triton configuration selected by Forge.

---
## 7. Live Demo: Run Hyperloom

The final two cells are the operational cells for a fresh run:

1. **Startup cell**: writes and executes a shell script that prepares the environment, runs install/preflight checks, launches Hyperloom in the background, and writes a latest-run pointer.
2. **Monitor cell**: reads that pointer, tails the log, and summarizes `state.json` until the process exits or the monitor window ends.

The cells do not print API keys or environment file contents. If your environment requires additional credentials or operator-specific setup, provide those file paths through `HYPERLOOM_OPERATOR_ENV_FILES` before running the startup cell.

In [ ]:
from pathlib import Path
import subprocess

RUN_DIR = Path("/workspace/hyperloom/optimizer_runs")
RUN_DIR.mkdir(parents=True, exist_ok=True)
SCRIPT_PATH = RUN_DIR / "start_qwen3_vllm_forge.sh"

script = r'''#!/usr/bin/env bash
set -Eeuo pipefail
set +x

export REPO_ROOT="${REPO_ROOT:-/workspace/projects/Hyperloom}"
export USER_DATA_PATH="${USER_DATA_PATH:-/workspace/hyperloom}"
export PYTHON="${PYTHON:-python3}"
PYTHON_BIN="$(command -v "$PYTHON" || true)"
if [ -n "$PYTHON_BIN" ]; then
  export PYTHON="$PYTHON_BIN"
fi
export HYPERLOOM_NOTEBOOK_PYTHON="$PYTHON"
python3() { "$HYPERLOOM_NOTEBOOK_PYTHON" "$@"; }
python() { "$HYPERLOOM_NOTEBOOK_PYTHON" "$@"; }
pip3() { "$HYPERLOOM_NOTEBOOK_PYTHON" -m pip "$@"; }
pip() { "$HYPERLOOM_NOTEBOOK_PYTHON" -m pip "$@"; }
export -f python3 python pip3 pip
export NPM_CONFIG_PREFIX="${NPM_CONFIG_PREFIX:-$USER_DATA_PATH/runtime/npm-global}"
mkdir -p "$NPM_CONFIG_PREFIX"
npm() {
  if [ "$#" -ge 4 ] && [ "$1" = "config" ] && [ "$2" = "set" ] && [ "$3" = "prefix" ] && [ "$4" = "/usr/local" ]; then
    command npm config set prefix "$NPM_CONFIG_PREFIX"
  else
    command npm "$@"
  fi
}
export -f npm
export NODE_PATH="$NPM_CONFIG_PREFIX/lib/node_modules:${NODE_PATH:-}"
export PATH="$NPM_CONFIG_PREFIX/bin:$(dirname "$PYTHON"):$PATH"

export MODEL="${MODEL:-/workspace/models/Qwen-Qwen3-30B-A3B}"
export MODEL_PATH="$MODEL"
export FRAMEWORK="${FRAMEWORK:-vllm}"
export GPU_TYPE="${GPU_TYPE:-mi300x}"
export TP="${TP:-1}"
export CONC="${CONC:-16}"
export ISL="${ISL:-1024}"
export OSL="${OSL:-1024}"
export MAX_MODEL_LEN="${MAX_MODEL_LEN:-6144}"
export PRECISION="${PRECISION:-bf16}"
export TARGET_GAIN="${TARGET_GAIN:-10}"
export MAX_HOURS="${MAX_HOURS:-1}"
export MODEL_CLASS="${MODEL_CLASS:-moe_swa}"
export TICK_INTERVAL_SEC="${TICK_INTERVAL_SEC:-30}"
export GPU_SPECIALIST_CAPACITY="${GPU_SPECIALIST_CAPACITY:-0}"

export HYPERLOOM_KERNEL_AGENT_ROOT="$REPO_ROOT/kernel-agent"
export KERNEL_AGENT_ROOT="$HYPERLOOM_KERNEL_AGENT_ROOT"
export HYPERLOOM_TRACE_ANALYSIS_ROUTE="${HYPERLOOM_TRACE_ANALYSIS_ROUTE:-deterministic}"
export KERNEL_OPT_BACKEND_ORDER="${KERNEL_OPT_BACKEND_ORDER:-forge}"
export GEMM_TUNING_BACKEND="${GEMM_TUNING_BACKEND:-forge}"

export TRACELENS_ROOT="${TRACELENS_ROOT:-/tmp/hyperloom/open-source-repos/TraceLens}"
export FORGE_PATH="${FORGE_PATH:-/workspace/projects/KernelForge}"
export FORGE_GEMM_TUNE_ROOT="${FORGE_GEMM_TUNE_ROOT:-/workspace/projects/KernelForge/src/forge_gemm_tune}"
export NODE_TLS_REJECT_UNAUTHORIZED="${NODE_TLS_REJECT_UNAUTHORIZED:-0}"
export RAY_VERSION="${RAY_VERSION:-2.45.0}"

export AITER_JIT_DIR="${AITER_JIT_DIR:-$HOME/.aiter/jit}"
export ROCM_PATH="${ROCM_PATH:-/opt/rocm}"
export HIP_PATH="${HIP_PATH:-/opt/rocm}"
export LIBRARY_PATH="/opt/rocm/lib:/opt/python/lib/python3.13/site-packages/_rocm_sdk_devel/lib:${LIBRARY_PATH:-}"
export LD_LIBRARY_PATH="/opt/rocm/lib:/opt/python/lib/python3.13/site-packages/_rocm_sdk_devel/lib:${LD_LIBRARY_PATH:-}"

if [ -d "$FORGE_GEMM_TUNE_ROOT" ]; then
  export PYTHONPATH="$FORGE_GEMM_TUNE_ROOT:$FORGE_PATH:${PYTHONPATH:-}"
fi

export RUN_DIR="${RUN_DIR:-$USER_DATA_PATH/optimizer_runs}"
mkdir -p "$RUN_DIR" "$AITER_JIT_DIR"
cd "$REPO_ROOT"

echo "--- optional operator environment ---"
if [ -n "${HYPERLOOM_OPERATOR_ENV_FILES:-}" ]; then
  for env_file in ${HYPERLOOM_OPERATOR_ENV_FILES}; do
    if [ -f "$env_file" ]; then
      echo "sourcing operator env file: $env_file"
      source "$env_file"
    else
      echo "WARNING: operator env file not found: $env_file"
    fi
  done
fi

echo "--- active process guard ---"
existing="$(pgrep -af 'vllm.entrypoints|sglang.launch_server|Magpie|inference_optimizer .*optimize|inference_optimizer.cli .*optimize' || true)"
if [ -n "$existing" ]; then
  echo "ERROR: Existing serving or optimizer process detected. Stop it before launching a workshop run."
  echo "$existing"
  exit 2
fi

echo "--- install/preflight assets ---"
bash "$REPO_ROOT/inference_optimizer/scripts/install.sh"

for env_file in "$USER_DATA_PATH/runtime/local-setup.env.sh" "$USER_DATA_PATH/runtime/kernel-agent.env.sh"; do
  if [ -f "$env_file" ]; then
    source "$env_file"
  else
    echo "WARNING: expected runtime env file is missing: $env_file"
  fi
done

echo "--- GPU occupancy snapshot ---"
if command -v rocm-smi >/dev/null 2>&1; then
  rocm-smi --showuse --showpidgpus || true
else
  echo "rocm-smi is not available in this shell; continuing."
fi

echo "--- Forge preflight ---"
command -v forge-gemm-tune || true
"$PYTHON" - <<'PY'
import importlib.util
import os
import pathlib
import shutil

print("forge_gemm_tune_cli=", shutil.which("forge-gemm-tune"))
print("forge_gemm_tune_import=", importlib.util.find_spec("forge_gemm_tune") is not None)
for name in ("FORGE_PATH", "FORGE_GEMM_TUNE_ROOT", "KERNEL_OPT_BACKEND_ORDER", "GEMM_TUNING_BACKEND"):
    value = os.environ.get(name, "")
    print(f"{name}={value}")
    if value and name.endswith(("PATH", "ROOT")):
        p = pathlib.Path(value)
        print(f"{name}_exists={p.exists()} is_dir={p.is_dir()}")
PY

if ! command -v forge-gemm-tune >/dev/null 2>&1; then
  set +e
  "$PYTHON" - <<'PY'
import importlib.util
import sys
sys.exit(0 if importlib.util.find_spec("forge_gemm_tune") else 1)
PY
  forge_import_status=$?
  set -e
  if [ "$forge_import_status" -ne 0 ] && [ ! -d "$FORGE_GEMM_TUNE_ROOT" ]; then
    echo "ERROR: Forge is not available. Set FORGE_PATH/FORGE_GEMM_TUNE_ROOT or install forge-gemm-tune."
    exit 3
  fi
fi

echo "--- Hyperloom CLI preflight ---"
"$PYTHON" -m inference_optimizer.cli optimize --help >/dev/null
test -d "$MODEL" && echo "model exists: $MODEL"
test -d "$TRACELENS_ROOT" && echo "TraceLens exists: $TRACELENS_ROOT" || true
ray status | sed -n '1,40p' || true

export RUN_TAG="${RUN_TAG:-qwen3-30b-a3b-vllm-forge-tp${TP}-conc${CONC}-isl${ISL}-osl${OSL}-$(date +%Y%m%d_%H%M%S)}"
export LOG_PATH="$RUN_DIR/run_${RUN_TAG}.log"
export LAUNCH_INFO_FILE="$RUN_DIR/launch_${RUN_TAG}.json"
export PID_FILE="$RUN_DIR/pid_${RUN_TAG}.txt"
export LATEST_POINTER="$RUN_DIR/latest_qwen3_vllm_forge_launch.json"

cmd=(
  "$PYTHON" -m inference_optimizer.cli --verbose optimize
  --model "$MODEL"
  --framework "$FRAMEWORK"
  --gpu-type "$GPU_TYPE"
  --tp "$TP"
  --conc "$CONC"
  --isl "$ISL"
  --osl "$OSL"
  --max-model-len "$MAX_MODEL_LEN"
  --precision "$PRECISION"
  --target-gain "$TARGET_GAIN"
  --max-hours "$MAX_HOURS"
  --tick-interval-sec "$TICK_INTERVAL_SEC"
  --model-class "$MODEL_CLASS"
  --gpu-specialist-capacity "$GPU_SPECIALIST_CAPACITY"
  --launch-info-file "$LAUNCH_INFO_FILE"
)

if [ -n "${EXTRA_OPTIMIZER_ARGS:-}" ]; then
  read -r -a extra_args <<< "$EXTRA_OPTIMIZER_ARGS"
  cmd+=("${extra_args[@]}")
fi

printf '%q ' "${cmd[@]}" > "$RUN_DIR/cmd_${RUN_TAG}.txt"
echo >> "$RUN_DIR/cmd_${RUN_TAG}.txt"

echo "--- launching Hyperloom ---"
setsid nohup "${cmd[@]}" > "$LOG_PATH" 2>&1 < /dev/null &
echo $! > "$PID_FILE"
PID="$(cat "$PID_FILE")"
export PID

sleep 20

echo "--- initial health check ---"
ps -p "$PID" -o pid,ppid,stat,etime,%cpu,%mem,cmd || true
tail -n 120 "$LOG_PATH" || true
ray status | sed -n '1,40p' || true

"$PYTHON" - <<'PY'
import json
import os
from pathlib import Path

payload = {
    "run_tag": os.environ["RUN_TAG"],
    "pid": int(os.environ["PID"]),
    "log_path": os.environ["LOG_PATH"],
    "launch_info_file": os.environ["LAUNCH_INFO_FILE"],
    "pid_file": os.environ["PID_FILE"],
    "cmd_file": str(Path(os.environ["RUN_DIR"]) / f"cmd_{os.environ['RUN_TAG']}.txt"),
    "framework": os.environ["FRAMEWORK"],
    "conc": int(os.environ["CONC"]),
    "target_gain": float(os.environ["TARGET_GAIN"]),
    "max_hours": float(os.environ["MAX_HOURS"]),
    "model": os.environ["MODEL"],
}
launch_info = Path(os.environ["LAUNCH_INFO_FILE"])
if launch_info.exists():
    try:
        payload["hyperloom_launch_info"] = json.loads(launch_info.read_text())
    except Exception as exc:
        payload["launch_info_parse_error"] = str(exc)
Path(os.environ["LATEST_POINTER"]).write_text(json.dumps(payload, indent=2) + "\n")
print(json.dumps(payload, indent=2))
PY

echo "--- launch paths ---"
echo "RUN_TAG=$RUN_TAG"
echo "PID=$PID"
echo "LOG_PATH=$LOG_PATH"
echo "LAUNCH_INFO_FILE=$LAUNCH_INFO_FILE"
echo "PID_FILE=$PID_FILE"
echo "LATEST_POINTER=$LATEST_POINTER"
'''

SCRIPT_PATH.write_text(script)
SCRIPT_PATH.chmod(0o755)
print(f"Wrote startup script: {SCRIPT_PATH}")

result = subprocess.run(["bash", str(SCRIPT_PATH)], text=True)
if result.returncode != 0:
    raise SystemExit(f"Startup script failed with exit code {result.returncode}")

In [ ]:
from pathlib import Path
import datetime as dt
import json
import os
import shlex
import subprocess
import time

RUN_DIR = Path(os.environ.get("HYPERLOOM_RUN_DIR", "/workspace/hyperloom/optimizer_runs"))
LATEST_POINTER = RUN_DIR / "latest_qwen3_vllm_forge_launch.json"
MONITOR_INTERVAL_SEC = int(os.environ.get("MONITOR_INTERVAL_SEC", "300"))
MONITOR_MAX_MINUTES = float(os.environ.get("MONITOR_MAX_MINUTES", "75"))
TAIL_LINES = int(os.environ.get("MONITOR_TAIL_LINES", "160"))


def load_latest():
    if LATEST_POINTER.exists():
        return json.loads(LATEST_POINTER.read_text())
    launch_files = sorted(RUN_DIR.glob("launch_qwen3-30b-a3b-vllm-forge-*.json"), key=lambda p: p.stat().st_mtime)
    if not launch_files:
        raise FileNotFoundError(f"No latest pointer or launch_*.json found under {RUN_DIR}")
    return {"launch_info_file": str(launch_files[-1])}


def is_running(pid: int) -> bool:
    return subprocess.run(["bash", "-lc", f"kill -0 {int(pid)} 2>/dev/null"], check=False).returncode == 0


def run_shell(command: str):
    subprocess.run(["bash", "-lc", command], check=False)


def read_launch_info(info: dict) -> dict:
    path = Path(info.get("launch_info_file", ""))
    if path.exists():
        try:
            return json.loads(path.read_text())
        except Exception as exc:
            return {"parse_error": str(exc), "path": str(path)}
    nested = info.get("hyperloom_launch_info")
    return nested if isinstance(nested, dict) else {}


def summarize_state(session_dir: Path):
    state_path = session_dir / "state.json"
    if not state_path.exists():
        print(f"state.json not available yet: {state_path}")
        return
    state = json.loads(state_path.read_text())
    current_best = state.get("current_best") or {}
    fields = {
        "phase": state.get("phase") or state.get("last_phase"),
        "stop_reason": state.get("stop_reason"),
        "baseline_tput": state.get("baseline_tput"),
        "cumulative_gain": state.get("cumulative_gain"),
        "cumulative_gain_validated": state.get("cumulative_gain_validated"),
        "current_best_action": current_best.get("action"),
        "current_best_tput": current_best.get("tput"),
        "current_best_variant": current_best.get("variant_name"),
    }
    print(json.dumps(fields, indent=2))


info = load_latest()
launch_info = read_launch_info(info)
pid = int(info.get("pid") or Path(info["pid_file"]).read_text().strip())
log_path = Path(info.get("log_path", ""))
session_dir = Path(launch_info.get("session_dir", "")) if launch_info.get("session_dir") else None

print("Monitoring run:")
print(json.dumps({**info, "hyperloom_launch_info": launch_info}, indent=2))

deadline = time.time() + MONITOR_MAX_MINUTES * 60
while True:
    now = dt.datetime.now().isoformat(timespec="seconds")
    running = is_running(pid)
    print(f"\n===== {now} pid={pid} running={running} =====")
    run_shell(f"ps -p {pid} -o pid,ppid,stat,etime,%cpu,%mem,cmd || true")

    launch_info = read_launch_info(info)
    if launch_info.get("session_dir"):
        session_dir = Path(launch_info["session_dir"])
        print(f"session_dir={session_dir}")
        summarize_state(session_dir)
    else:
        print("session_dir not emitted yet")

    if log_path.exists():
        print(f"--- tail -n {TAIL_LINES} {log_path} ---")
        run_shell(f"tail -n {TAIL_LINES} {shlex.quote(str(log_path))} || true")
    else:
        print(f"log not available yet: {log_path}")

    if not running:
        print("Process has exited.")
        break
    if time.time() >= deadline:
        print(f"Monitor window ended after {MONITOR_MAX_MINUTES} minutes; rerun this cell to continue.")
        break
    time.sleep(MONITOR_INTERVAL_SEC)

if session_dir:
    report = session_dir / "reports" / "final.md"
    current_setting = session_dir / "current_setting.sh"
    print(f"\nFinal report: {report} exists={report.exists()}")
    print(f"Current setting: {current_setting} exists={current_setting.exists()}")